Descarrego les campanyes publicitàries que s'han realitzat des del 2000, les qual es troben a la ruta "https://residus.gencat.cat/ca/ambits_dactuacio/sensibilitzacio/campanyes/index.html"
Al no tenir cap API i estar la web protegida, i no poder fer WebScraping, obtenim les dades generant un PDF i amb OCR passar la informació a un Word, i aquest és el que carreguem amb Python per tal d'obtenir la informació desitjada.

In [ ]:
from docx import Document
import pandas as pd
import re

def extreure_de_word(ruta_fitxer):
    try:
        doc = Document(ruta_fitxer)
        print("Document obert correctament #02AD3A")
        
        dades = []
        paragrafs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        
        for i, text in enumerate(paragrafs):
            # Busquem les línies que contenen el títol de la campanya
            if "Campanya" in text or "RECI-CLAR" in text or "Kit digital" in text:
                campanya = text.replace('Campanya "', '').replace('"', '')
                any_campanya = "N/A"
                
                # Busquem l'any en els següents 2 paràgrafs (format 202X)
                for j in range(1, 3):
                    if i + j < len(paragrafs):
                        seguent = paragrafs[i + j]
                        if re.search(r'\d{4}', seguent):
                            any_campanya = seguent
                            break
                
                dades.append({"Campanya": campanya, "Any": any_campanya})
        
        return pd.DataFrame(dades).drop_duplicates()

    except Exception as e:
        print(f"Error #A80000: {e}")
        return None

# La teva ruta
ruta = r"C:\Users\kytus\Downloads\campanyes_reciclatge.docx"

campanyes = extreure_de_word(ruta)

if campanyes is not None and not campanyes.empty:
    print("\nTaula finalitzada:")
    print(campanyes.to_string(index=False))
else:
    print("\n#A80000: El Word sembla buit o el contingut és una imatge dins del document.")

In [ ]:
campanyes = campanyes.drop(index=[18,26,33,34])

In [ ]:
pd.set_option('display.max_colwidth', None)  # No talla el contingut de les cel·les
pd.set_option('display.max_rows', None)      # Mostra totes les files
pd.set_option('display.width', 1000)         # Amplada total de la línia perquè no salti
campanyes

In [ ]:
import pandas as pd
import re

def expandir_anys_v2(row):
    text = str(row['Any'])
    # Buscamos todos los números de 4 dígitos en el string
    anys_trobats = re.findall(r'\d{4}', text)
    
    if len(anys_trobats) >= 2:
        # Si encuentra dos o más (ej: '2024 2025' o '2024-2025')
        inici = int(anys_trobats[0])
        fi = int(anys_trobats[-1]) # Cogemos el último por si hay rangos largos
        return list(range(inici, fi + 1))
    elif len(anys_trobats) == 1:
        # Si solo hay un año (ej: '2024')
        return [int(anys_trobats[0])]
    else:
        # Si no encuentra nada o el formato es erróneo (#A80000)
        return []

# Aplicamos la limpieza
campanyes['Any_List'] = campanyes.apply(expandir_anys_v2, axis=1)

# "Explotamos" la lista para tener una fila por año
campanyes_neta = campanyes.explode('Any_List')

# Eliminamos filas que no tengan año y convertimos a int
campanyes_neta = campanyes_neta.dropna(subset=['Any_List'])
campanyes_neta['Any_List'] = campanyes_neta['Any_List'].astype(int)



# Renombrar para el merge final
# campanyes_neta = campanyes_neta.rename(columns={'Any_List': 'Any'})

print("¡Conversión OK (#02AD3A)!")
print(campanyes_neta)

In [ ]:
# Eliminem columna Any
campanyes_neta = campanyes_neta.drop(columns='Any')




In [ ]:
# Renombrem Any_list
campanyes_neta = campanyes_neta.rename(columns={'Any_List': 'Any'})
campanyes_neta

Amb WebScrap obtenim les dades de municipis que realitzen el servei de Porta a Porta

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def obtenir_municipis_pap():
    url = "https://portaaporta.cat/ca/municipis.php"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code == 200:
            print("Connexió establerta #02AD3A")
            
            # Busquem taules dins de l'HTML
            # pandas té una funció màgica per llegir taules HTML directament
            taules = pd.read_html(response.text)
            
            if taules:
                # Normalment la taula de municipis és la primera o única de la pàgina
                df = taules[0]
                
                # Neteja bàsica: treure columnes buides si n'hi ha
                df = df.dropna(how='all', axis=1)
                
                return df
            else:
                print("No s'han trobat taules a l'HTML #A80000")
                return None
        else:
            print(f"Error de servidor: {response.status_code} #A80000")
            return None

    except Exception as e:
        print(f"Error en l'scraping: {e} #A80000")
        return None

# Executar
porta_a_porta = obtenir_municipis_pap()

if porta_a_porta is not None:
    print("\nTaula de Municipis (Primers 10):")
    print(porta_a_porta.head(10).to_string(index=False))
    
    # Opcional: Guardar a Excel
    # porta_a_porta.to_excel("municipis_pap.xlsx", index=False)
else:
    print("\nL'scraping ha fallat. Probablement la taula es carrega via AJAX.")

In [ ]:
# modificar separador milers i decimals
porta_a_porta['Cens 2024'] = porta_a_porta['Cens 2024'].str.replace('.', '')
porta_a_porta['Cens 2024'] = porta_a_porta['Cens 2024'].str.replace(',','.')
porta_a_porta['habitants servits amb el PaP'] = porta_a_porta['habitants servits amb el PaP'].str.replace('.', '')
porta_a_porta['habitants servits amb el PaP'] = porta_a_porta['habitants servits amb el PaP'].str.replace(',','.')

In [ ]:
# Canviar tipo de dades dels camps, passant a date i float
porta_a_porta['Data d\'inici'] = pd.to_datetime(porta_a_porta['Data d\'inici'])
porta_a_porta['Cens 2024'] = porta_a_porta['Cens 2024'].astype(float)
porta_a_porta['habitants servits amb el PaP'] = porta_a_porta['habitants servits amb el PaP'].astype(float)


In [ ]:
porta_a_porta.info()

In [ ]:
import pandas as pd
residus_municipals = pd.read_csv(r"C:\Users\kytus\Documents\Bootcamp\Projecte\residus\Estadístiques_de_residus_municipals_20260122.csv")
tipus_residus = pd.read_csv(r"C:\Users\kytus\Documents\Bootcamp\Projecte\residus\Llista_de_residus_i_la_seva_recollida_selectiva_20260122.csv")


In [ ]:
residus_municipals = residus_municipals.rename(columns={'Municipi':'municipi',
                                                        'Any': 'any'})

residus_municipals['Kg/hab/any recollida selectiva'] = residus_municipals['Kg/hab/any recollida selectiva'].str.replace('.','')
residus_municipals['Kg/hab/any recollida selectiva'] = residus_municipals['Kg/hab/any recollida selectiva'].str.replace(',','.')
residus_municipals['Kg/hab/any recollida selectiva'] = residus_municipals['Kg/hab/any recollida selectiva'].astype(float)

Llistar i Carregar els CSV descarregats de IDESCAT

In [ ]:
import os
import pandas as pd

directorio = r"C:\Users\kytus\Documents\Bootcamp\Projecte\poblacio"
archivos_csv = [f for f in os.listdir(directorio) if f.endswith('.csv')]
archivos = [archivo.replace('.csv', '') for archivo in archivos_csv]
print(archivos)

In [ ]:
dfs = {}
for tabla in archivos:
    dfs[tabla] = pd.read_csv(f"{directorio}\{tabla}.csv", sep=';')

for nombre in archivos:
    globals()[f'df_{nombre}'] = dfs[nombre]   

Crear taula reestructurada de superficie, per tenir columnes endreçades

In [ ]:
superficie_municipis = dfs['superficie_poblacio'].pivot(
    index='municipi',
    columns='col',
    values='value'
).reset_index()


superficie_municipis['Superfície (km²)']= superficie_municipis['Superfície (km²)'].str.replace(',','.').astype(float)


Agrupar tots els anys de població en un sol DF

In [ ]:
poblacio_municipis = pd.concat([valor for clau, valor in dfs.items() if clau.startswith('poblacio')], axis=0, ignore_index=True)


Detectar els que estat es difernet, perque son els que no tenen valor a poblacio

In [ ]:

filtro_estat = poblacio_municipis[poblacio_municipis['estat'].str.contains('..', case=False, na=False)]

print(f"Se han encontrado {len(filtro_estat)} filas:")
print(filtro_estat['estat'].unique())


Eliminar aquests registres amb estat = ..

In [ ]:
poblacio_municipis.drop(poblacio_municipis[poblacio_municipis['estat']== '..'].index, inplace=True)

convertir poblacio a int

In [ ]:
poblacio_municipis['valor'] = poblacio_municipis['valor'].astype('int')

Modificar columnes sexe i total

In [ ]:
poblacio_municipis = poblacio_municipis.pivot_table(
    index=['any', 'municipi'], 
    columns='sexe', 
    values='valor'
).reset_index()



Crear Dataframe poblacio Catalunya i eliminar de municipis la poblacio total de Catalunya

In [ ]:
poblacio_Catalunya = poblacio_municipis[poblacio_municipis['municipi'].str.contains('Catalunya', case=False, na=False)].copy()

# poblacio_Catalunya = poblacio_Catalunya.pivot_table(
#     index=['any', 'municipi'], 
#     columns='sexe', 
#     values='valor'
# ).reset_index()

poblacio_Catalunya

Eliminar Catalunya com a municipi

In [ ]:
poblacio_municipis.drop(poblacio_municipis[poblacio_municipis['municipi']== 'Catalunya'].index, inplace=True)

Afegir camp superficie i calcular densitat poblacio

In [ ]:
# afegir camps comarca i superficie
poblacio_municipis = poblacio_municipis.merge(superficie_municipis[['municipi','Comarca','Superfície (km²)']], on='municipi', how='left')


In [ ]:
#calcular densitat de poblacio
poblacio_municipis['densitat_poblacio']= (poblacio_municipis['total']/poblacio_municipis['Superfície (km²)']).round(2)



Repetir proces amb les taules de renta. Agrupar en un sol DF

In [ ]:
renda_municipis = pd.concat([valor for clau, valor in dfs.items() if clau.startswith('renta')], axis=0, ignore_index=True)
renda_municipis['any'].astype('Int64')
renda_municipis['valor'] = renda_municipis['valor'].str.replace(',','.').astype(float)
renda_municipis.info()

In [ ]:
renda_municipis = renda_municipis.pivot_table(
    index=['any', 'municipi'], 
    columns=['concepte','indicador'], 
    values='valor'
).reset_index()

Multipliquem per 1000 les dades de renda del 2000 al 2016

In [ ]:
# Definimos la columna con sus dos niveles
columna_renta = ('renda familiar disponible bruta', 'per habitant (€)')

# Aplicamos la multiplicación filtrando por el año
renda_municipis.loc[
    (renda_municipis['any'] >= 2000) & (renda_municipis['any'] <= 2016), 
    columna_renta
] *= 1000

# Comprobamos el cambio
print(renda_municipis.head())

Taules:
- renda_municipis = renda per municipi del 2000 al 2022
- poblacio_municipis = població, superficie i densitat de poblacio per mmunicidpi del 2000 al 2024
- porta_a_porta = data incii servei de recollida porta a porta i municipis adherits
- residus_municipals = qunatitat de residus generada per municipi i de quina categoria
- tipus_residus = categories dels residus que es recullen
- campanyes = anuncis fets per incentivar i educar en el reciclatge


In [ ]:
residus_municipals.sort_values('Kg/hab/any recollida selectiva', ascending=False)

In [ ]:
residus_poblacio_densitat = residus_municipals.merge(right=poblacio_municipis[['municipi','dones','homes','total','Superfície (km²)','densitat_poblacio']], how='left', on='municipi')

In [ ]:
residus_poblacio_densitat['Kg/hab/any recollida selectiva'].info()

In [ ]:
residus_poblacio_densitat.keys()

In [ ]:
import pandas as pd

# 1. Cargar los archivos
municipis_comarca = pd.read_csv(r'C:\Users\kytus\Documents\Bootcamp\Projecte\idescat-aec-15903-1.csv', sep=';')
provincies_comarca = pd.read_csv(r'C:\Users\kytus\Documents\Bootcamp\Projecte\idescat-aec-15902-1.csv', sep=';')

# 2. Extraer Comarca y Codi del archivo de municipios (15903)
# Filtramos por las dos etiquetas que queremos
municipis_base = municipis_comarca[municipis_comarca['col'].isin(['Comarca', 'Codi'])]

# Pivotamos para que 'Comarca' y 'Codi' sean columnas individuales
df_municipis = municipis_base.pivot(index='row', columns='col', values='value').reset_index()
df_municipis.columns = ['Municipi', 'Codi_Municipi', 'Comarca']

# 3. Extraer la relación Comarca -> Província (del archivo 15902)
comarques = provincies_comarca[provincies_comarca['col'] == 'Províncies a les quals pertanyen els municipis'][['row', 'value']]
comarques.columns = ['Comarca', 'Província']

# 4. Unir todo en un solo DataFrame
provincies_municipi = pd.merge(df_municipis, comarques, on='Comarca', how='left')

# Reordenar columnas para que quede más limpio
provincies_municipi = provincies_municipi[['Codi_Municipi', 'Municipi', 'Comarca', 'Província']]

# Ver el resultado
print(provincies_municipi.head())

# Si quieres guardarlo:
# df_final.to_csv('dades_municipis_complets.csv', index=False, sep=';', encoding='utf-8-sig')

Exportar DataFrames a CSV, per treballar de forma mes agil

In [ ]:
provincies_municipi.to_csv(r'DataFrames\provincies_municipi.csv')
renda_municipis.to_csv(r'DataFrames\renda_municipis.csv')
poblacio_Catalunya.to_csv(r'DataFrames\poblacio_Catalunya.csv')
poblacio_municipis.to_csv(r'DataFrames\poblacio_municipìs.csv')
porta_a_porta.to_csv(r'DataFrames\porta_a_porta.csv')
residus_municipals.to_csv(r'DataFrames\residus_municipals.csv')
tipus_residus.to_csv(r'DataFrames\tipus_residus.csv')
campanyes_neta.to_csv(r'DataFrames\campanyes_neta.csv')